# EMG Segraw → Moments Feature Engineering

Pipeline:
1. Load raw segmented EMG CSVs into nested dict
2. BPF at 20-450 Hz
3. Per-gesture std normalisation
4. Khushaba spectral moments FE (5 features × 16 EMG channels = 80 features per gesture)
5. Global feature-matrix std normalisation
6. Save to CSV — one row per gesture trial

Mirror of `ppdsegraw_feature_engineering.ipynb` for EMG, with two changes:
- `modalities=['E']` in `load_segraw_data`

Output file: `moments_segraw_allEMG.csv`

In [ ]:
import numpy as np
import pandas as pd
import time
import json

# All shared preprocessing + FE functions live here
from shared_processing import (
    load_segraw_data,
    apply_filter_to_nested_dict,
    normalize_gestures_by_std_any_channels,
    create_khushaba_spectralmomentsFE_vectors,
    normalize_whole_dataset_features,
)

## 0. Config — set your paths here

In [ ]:
# ── UPDATE THESE TO YOUR MACHINE ──────────────────────────────────────────
# C:\Users\kdmen\Box\Yamagami Lab\Data\2024_UIST_dataset\upload\segmented_raw_data
RAW_DATA_DIR   = r"C:\Users\kdmen\Box\Yamagami Lab\Data\2024_UIST_dataset\upload\segmented_raw_data"
SAVE_DIR       = r"C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_2024"
# ──────────────────────────────────────────────────────────────────────────

# Intermediate save (nested dict after loading, before FE) — optional but
# saves ~5 min if you need to re-run FE without reloading CSVs
RAW_DICT_SAVE_PATH      = SAVE_DIR + r"\segraw_EMG_allgestures_allusers.json"
PPD_DICT_SAVE_PATH      = SAVE_DIR + r"\ppdsegraw_allEMG.json"
MOMENTS_CSV_SAVE_PATH   = SAVE_DIR + r"\moments_segraw_allEMG.csv"

# Participant lists — identical to your EMG pipeline
pIDs_impaired   = ['P102','P103','P104','P105','P106','P107','P108','P109','P110','P111',
                   'P112','P114','P115','P116','P118','P119','P121','P122','P123','P124',
                   'P125','P126','P127','P128','P131','P132']
pIDs_unimpaired = ['P004','P005','P006','P008','P010','P011']
pIDs_both       = pIDs_impaired + pIDs_unimpaired

# NOTE: 2000 Hz was for EMG
# 148 Hz was for IMU
FS = 2000  # Nominal sampling rate stored in the raw files (used for moment calculations)

## 1. Load raw EMG CSVs

> Same `load_segraw_data` as the EMG pipeline
> Each gesture ends up as a list of 16 channels × ~6000 timepoints.

In [ ]:
print("Loading raw EMG data...")
start = time.time()

nested_dict = load_segraw_data(
    pIDs      = pIDs_both,
    data_dir_path = RAW_DATA_DIR,
    modalities    = ["E"],          
    expt_types    = ["experimenter-defined"],
    num_emg_sensors = 16,
)

print(f"Done in {time.time()-start:.1f}s")
print(f"Participants loaded: {list(nested_dict.keys())}")

In [ ]:
# Quick sanity check — should be 72 channels (12 sensors × 6 axes) --> It might think there's 15 sensors since the highest sensor channel was labeled 15...
sample_pid     = list(nested_dict.keys())[0]
sample_gesture = list(nested_dict[sample_pid].keys())[0]
sample_trial   = list(nested_dict[sample_pid][sample_gesture].keys())[0]
sample_data    = nested_dict[sample_pid][sample_gesture][sample_trial]["EMG"]

print(f"Sample: {sample_pid} / {sample_gesture} / trial {sample_trial}")
print(f"  n_channels  : {len(sample_data)}   (expected 16)")
print(f"  n_timepoints: {len(sample_data[0])}  (expected ~6000)")

In [ ]:
# Optional: save the raw nested dict so you don't have to reload CSVs next time
print("Saving raw EMG nested dict...")
import json
with open(RAW_DICT_SAVE_PATH, 'w') as f:
    json.dump(nested_dict, f)
print(f"Saved → {RAW_DICT_SAVE_PATH}")

## 2. Normalise and BPF, mean-subtract, then per-gesture std=1

> We do mean subtraction + per-gesture std normalisation and BPFing

In [ ]:
start = time.time()

ms_dict = apply_filter_to_nested_dict(
    nested_dict,
    normalization_method = "MEANSUBTRACTION",
    already_BPFd         = False,  
)

print(f"Done in {time.time()-start:.1f}s")

In [ ]:
print("Per-gesture std normalisation...")
start = time.time()

# Use the any-channels variant so it handles 16 EMG channels automatically
ppd_dict = normalize_gestures_by_std_any_channels(ms_dict)

print(f"Done in {time.time()-start:.1f}s")

# Sanity check: std across flattened channels should be ~1
sample_data_ppd = ppd_dict[sample_pid][sample_gesture][sample_trial]["EMG"]
flat = [v for ch in sample_data_ppd for v in ch]
print(f"  Post-normalisation std (should be ~1.0): {np.std(flat):.4f}")

In [ ]:
# Optional: save the preprocessed dict
print("Saving preprocessed EMG dict...")
with open(PPD_DICT_SAVE_PATH, 'w') as f:
    json.dump(ppd_dict, f)
print(f"Saved → {PPD_DICT_SAVE_PATH}")

## 3. Khushaba Moments Feature Engineering

For each gesture trial, for each of the 16 EMG channels, we compute:
- `zero_order`        — total signal power (energy)
- `second_order`      — energy of rate-of-change (first derivative)
- `fourth_order`      — energy of jerk (second derivative)
- `sparsity`          — sqrt(m0*m4) / m2
- `irregularity_factor` — m2^2 / (m0*m4), normalised by waveform length

Result: **80 features per gesture trial** (16 channels × 5 features).
All degenerate values (e.g. near-constant channels) are handled by
`safe_log` — clipped to [-10, 10] rather than blowing up to ±inf.

In [ ]:
print("Computing Khushaba moments...")
start = time.time()

moments_df = create_khushaba_spectralmomentsFE_vectors(ppd_dict, fs=FS)

print(f"Done in {time.time()-start:.1f}s")
print(f"Shape: {moments_df.shape}   (expected: n_trials × 453 = n_trials × [3 meta + 450 features])")
moments_df.head()

In [ ]:
# Check for any surviving NaN or inf (should be zero with safe_log)
n_nan = moments_df.isna().sum().sum()
n_inf = np.isinf(moments_df.select_dtypes(include=np.number).values).sum()
print(f"NaN count: {n_nan}  |  Inf count: {n_inf}")

if n_nan > 0 or n_inf > 0:
    print("WARNING: Unexpected NaN/inf — imputing with column mean as fallback.")
    moments_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    feature_cols = [c for c in moments_df.columns if c not in ['Participant','Gesture_ID','Gesture_Num']]
    moments_df[feature_cols] = moments_df[feature_cols].fillna(moments_df[feature_cols].mean())
else:
    print("Clean — no NaN or inf values.")

## 4. Global feature-matrix normalisation

Divides every feature by the global standard deviation of the entire
feature matrix, so all 450 features are on the same scale.
This is the same final step as in the EMG pipeline.

In [ ]:
n_moments_df = normalize_whole_dataset_features(moments_df)

print(f"Shape after normalisation: {n_moments_df.shape}")
n_moments_df.head()

## 5. Save

In [ ]:
n_moments_df.to_csv(MOMENTS_CSV_SAVE_PATH, index=False, header=True)
print(f"Saved → {MOMENTS_CSV_SAVE_PATH}")
print(f"Final shape: {n_moments_df.shape}")
print(f"Columns: {list(n_moments_df.columns[:6])} ... {list(n_moments_df.columns[-3:])}")

## 6. Quick verification

In [ ]:
# Reload and spot-check
check_df = pd.read_csv(MOMENTS_CSV_SAVE_PATH)
print(f"Reloaded shape : {check_df.shape}")
print(f"Participants   : {sorted(check_df['Participant'].unique())}")
print(f"Gestures       : {sorted(check_df['Gesture_ID'].unique())}")
print(f"Trials per gesture per participant (should be 10):")
print(check_df.groupby(['Participant','Gesture_ID'])['Gesture_Num'].count().describe())

feature_cols = [c for c in check_df.columns if c not in ['Participant','Gesture_ID','Gesture_Num']]
print(f"\nFeature stats (post-normalisation):")
print(check_df[feature_cols].describe().loc[['mean','std','min','max']])